In [61]:
# Loading the cleaned data and sorting it by time so future matches do not affect past features

In [62]:
import pandas as pd
import numpy as np

#importing data set
df = pd.read_csv('data/deliveries_cleaned.csv')

#correcting date datatype
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(by=["date", "matchid", "inning", "over", "ball"])
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   matchid           260920 non-null  int64         
 1   inning            260920 non-null  int64         
 2   over_ball         260920 non-null  float64       
 3   over              260920 non-null  int64         
 4   ball              260920 non-null  int64         
 5   batting_team      260920 non-null  object        
 6   bowling_team      260920 non-null  object        
 7   batsman           260920 non-null  object        
 8   non_striker       260920 non-null  object        
 9   bowler            260920 non-null  object        
 10  batsman_runs      260920 non-null  int64         
 11  extras            260920 non-null  int64         
 12  iswide            260920 non-null  float64       
 13  isnoball          260920 non-null  float64       
 14  byes

,matchid,inning,over_ball,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,...,iswide,isnoball,byes,legbyes,penalty,dismissal_kind,player_dismissed,date,total_runs,is_wicket
0,335982,1,0.1,0,1,kolkata knight riders,royal challengers bangalore,sc ganguly,BB McCullum,p kumar,...,0.0,0.0,0.0,1.0,0.0,unknown,unknown,2008-04-18,2,0
1,335982,1,0.2,0,2,kolkata knight riders,royal challengers bangalore,bb mccullum,SC Ganguly,p kumar,...,0.0,0.0,0.0,0.0,0.0,unknown,unknown,2008-04-18,0,0
2,335982,1,0.3,0,3,kolkata knight riders,royal challengers bangalore,bb mccullum,SC Ganguly,p kumar,...,1.0,0.0,0.0,0.0,0.0,unknown,unknown,2008-04-18,1,0
3,335982,1,0.4,0,4,kolkata knight riders,royal challengers bangalore,bb mccullum,SC Ganguly,p kumar,...,0.0,0.0,0.0,0.0,0.0,unknown,unknown,2008-04-18,0,0
4,335982,1,0.5,0,5,kolkata knight riders,royal challengers bangalore,bb mccullum,SC Ganguly,p kumar,...,0.0,0.0,0.0,0.0,0.0,unknown,unknown,2008-04-18,0,0


In [63]:
# Converting ball-by-ball data into one row per player per match, which is what we are predicting

In [64]:
#batsman statistics per match
batting_match = df.groupby(
    ["matchid", "date", "batsman", "batting_team", "bowling_team"]
).agg(
    runs_scored=("batsman_runs", "sum"),
    balls_faced=("batsman_runs", "count"),
    fours=("batsman_runs", lambda x: (x == 4).sum()),
    sixes=("batsman_runs", lambda x: (x == 6).sum()),
    dismissed=("is_wicket", "max")
).reset_index()
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   matchid       16515 non-null  int64         
 1   date          16515 non-null  datetime64[ns]
 2   batsman       16515 non-null  object        
 3   batting_team  16515 non-null  object        
 4   bowling_team  16515 non-null  object        
 5   runs_scored   16515 non-null  int64         
 6   balls_faced   16515 non-null  int64         
 7   fours         16515 non-null  int64         
 8   sixes         16515 non-null  int64         
 9   dismissed     16515 non-null  int64         
dtypes: datetime64[ns](1), int64(6), object(3)
memory usage: 1.3+ MB


,matchid,date,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed
0,335982,2008-04-18,aa noffke,royal challengers bangalore,kolkata knight riders,9,12,1,0,1
1,335982,2008-04-18,b akhil,royal challengers bangalore,kolkata knight riders,0,2,0,0,1
2,335982,2008-04-18,bb mccullum,kolkata knight riders,royal challengers bangalore,158,77,10,13,0
3,335982,2008-04-18,cl white,royal challengers bangalore,kolkata knight riders,6,10,0,0,1
4,335982,2008-04-18,dj hussey,kolkata knight riders,royal challengers bangalore,12,12,1,0,1


In [65]:
# Sorting each player's matches in the correct order so career and form features use only past games

In [66]:
#sorting batting_match by batsman
batting_match = batting_match.sort_values(
    by=["batsman", "date", "matchid"]
).reset_index(drop=True)
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   matchid       16515 non-null  int64         
 1   date          16515 non-null  datetime64[ns]
 2   batsman       16515 non-null  object        
 3   batting_team  16515 non-null  object        
 4   bowling_team  16515 non-null  object        
 5   runs_scored   16515 non-null  int64         
 6   balls_faced   16515 non-null  int64         
 7   fours         16515 non-null  int64         
 8   sixes         16515 non-null  int64         
 9   dismissed     16515 non-null  int64         
dtypes: datetime64[ns](1), int64(6), object(3)
memory usage: 1.3+ MB


,matchid,date,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed
0,548346,2012-04-29,a ashish reddy,deccan chargers,mumbai indians,10,10,0,1,1
1,548352,2012-05-04,a ashish reddy,deccan chargers,chennai super kings,3,3,0,0,1
2,548359,2012-05-08,a ashish reddy,deccan chargers,kings xi punjab,8,8,1,0,1
3,548373,2012-05-18,a ashish reddy,deccan chargers,rajasthan royals,10,4,2,0,0
4,548376,2012-05-20,a ashish reddy,deccan chargers,royal challengers bangalore,4,5,0,0,1


In [67]:
# Calculating strike rate to capture how quickly a batsman is scoring, not just total runs

In [68]:
#adding strike rate of players
batting_match["strike_rate"] = (
    batting_match["runs_scored"] / batting_match["balls_faced"] * 100
).fillna(0)
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   matchid       16515 non-null  int64         
 1   date          16515 non-null  datetime64[ns]
 2   batsman       16515 non-null  object        
 3   batting_team  16515 non-null  object        
 4   bowling_team  16515 non-null  object        
 5   runs_scored   16515 non-null  int64         
 6   balls_faced   16515 non-null  int64         
 7   fours         16515 non-null  int64         
 8   sixes         16515 non-null  int64         
 9   dismissed     16515 non-null  int64         
 10  strike_rate   16515 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(6), object(3)
memory usage: 1.4+ MB


,matchid,date,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed,strike_rate
0,548346,2012-04-29,a ashish reddy,deccan chargers,mumbai indians,10,10,0,1,1,100.0
1,548352,2012-05-04,a ashish reddy,deccan chargers,chennai super kings,3,3,0,0,1,100.0
2,548359,2012-05-08,a ashish reddy,deccan chargers,kings xi punjab,8,8,1,0,1,100.0
3,548373,2012-05-18,a ashish reddy,deccan chargers,rajasthan royals,10,4,2,0,0,250.0
4,548376,2012-05-20,a ashish reddy,deccan chargers,royal challengers bangalore,4,5,0,0,1,80.0


In [69]:
# Counting how many matches a player has already played to represent experience before this match

In [70]:
# total matches played by batsman in entire career from 2008-2024
batting_match["career_matches"] = (
    batting_match.groupby("batsman").cumcount()
)

In [71]:
# Computing total career runs using only previous matches to avoid leaking current match information

In [72]:
# career runs (before current match)
batting_match["career_runs"] = (
    batting_match.groupby("batsman")["runs_scored"]
    .cumsum()
    .shift(1)
    .fillna(0)
)

In [73]:
# Calculating career average runs to measure long-term consistency rather than raw totals

In [74]:
# career average runs 
batting_match["career_avg_runs"] = np.where(
    batting_match["career_matches"] > 0,
    batting_match["career_runs"] / batting_match["career_matches"],
    0
)
batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   matchid          16515 non-null  int64         
 1   date             16515 non-null  datetime64[ns]
 2   batsman          16515 non-null  object        
 3   batting_team     16515 non-null  object        
 4   bowling_team     16515 non-null  object        
 5   runs_scored      16515 non-null  int64         
 6   balls_faced      16515 non-null  int64         
 7   fours            16515 non-null  int64         
 8   sixes            16515 non-null  int64         
 9   dismissed        16515 non-null  int64         
 10  strike_rate      16515 non-null  float64       
 11  career_matches   16515 non-null  int64         
 12  career_runs      16515 non-null  float64       
 13  career_avg_runs  16515 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int6

,matchid,date,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed,strike_rate,career_matches,career_runs,career_avg_runs
0,548346,2012-04-29,a ashish reddy,deccan chargers,mumbai indians,10,10,0,1,1,100.0,0,0.0,0.00
1,548352,2012-05-04,a ashish reddy,deccan chargers,chennai super kings,3,3,0,0,1,100.0,1,10.0,10.00
2,548359,2012-05-08,a ashish reddy,deccan chargers,kings xi punjab,8,8,1,0,1,100.0,2,13.0,6.50
3,548373,2012-05-18,a ashish reddy,deccan chargers,rajasthan royals,10,4,2,0,0,250.0,3,21.0,7.00
4,548376,2012-05-20,a ashish reddy,deccan chargers,royal challengers bangalore,4,5,0,0,1,80.0,4,31.0,7.75


In [75]:
# recent form features
# Adding recent form over the last 5 matches to capture short-term performance trends
# last 5 matches
batting_match["form_runs_last_5"] = (
    batting_match
    .groupby("batsman")["runs_scored"]
    .shift(1)
    .rolling(window=5, min_periods=1)
    .mean()
)
# Adding recent form over the last 10 matches to smooth out one-off good or bad performances
# last 10 matches
batting_match["form_runs_last_10"] = (
    batting_match
    .groupby("batsman")["runs_scored"]
    .shift(1)
    .rolling(window=10, min_periods=1)
    .mean()
)

In [76]:
# Measuring how a batsman usually performs against the current bowling team

In [77]:
# opponent strength (using the bowling_team column as opponent)
batting_match["avg_runs_vs_opponent"] = (
    batting_match
    .groupby(["batsman", "bowling_team"])["runs_scored"]
    .transform("mean")
)
# Defining the target variable as runs scored in the match, which the model is learning to predict
#target variable
batting_match["target_runs"] = batting_match["runs_scored"]

batting_match.info()
batting_match.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   matchid               16515 non-null  int64         
 1   date                  16515 non-null  datetime64[ns]
 2   batsman               16515 non-null  object        
 3   batting_team          16515 non-null  object        
 4   bowling_team          16515 non-null  object        
 5   runs_scored           16515 non-null  int64         
 6   balls_faced           16515 non-null  int64         
 7   fours                 16515 non-null  int64         
 8   sixes                 16515 non-null  int64         
 9   dismissed             16515 non-null  int64         
 10  strike_rate           16515 non-null  float64       
 11  career_matches        16515 non-null  int64         
 12  career_runs           16515 non-null  float64       
 13  career_avg_runs 

,matchid,date,batsman,batting_team,bowling_team,runs_scored,balls_faced,fours,sixes,dismissed,strike_rate,career_matches,career_runs,career_avg_runs,form_runs_last_5,form_runs_last_10,avg_runs_vs_opponent,target_runs
0,548346,2012-04-29,a ashish reddy,deccan chargers,mumbai indians,10,10,0,1,1,100.0,0,0.0,0.00,NaN,NaN,13.500000,10
1,548352,2012-05-04,a ashish reddy,deccan chargers,chennai super kings,3,3,0,0,1,100.0,1,10.0,10.00,10.00,10.00,15.000000,3
2,548359,2012-05-08,a ashish reddy,deccan chargers,kings xi punjab,8,8,1,0,1,100.0,2,13.0,6.50,6.50,6.50,12.333333,8
3,548373,2012-05-18,a ashish reddy,deccan chargers,rajasthan royals,10,4,2,0,0,250.0,3,21.0,7.00,7.00,7.00,12.333333,10
4,548376,2012-05-20,a ashish reddy,deccan chargers,royal challengers bangalore,4,5,0,0,1,80.0,4,31.0,7.75,7.75,7.75,11.000000,4


In [78]:
# Removing columns that directly reveal the target so the model stays leakage-free
# Filling remaining missing values so the model can train without errors
# Saving the final feature dataset so it can be reused for model training and evaluation

In [79]:
# removing leakage column
features_df = batting_match.drop(
    columns=[
        "runs_scored",
        "career_runs"
    ]
)
# final clean up
features_df = features_df.fillna(0)

#final checking and saving data
features_df.info()
features_df.to_csv("dataset_batting_features.csv", index=False)
features_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16515 entries, 0 to 16514
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   matchid               16515 non-null  int64         
 1   date                  16515 non-null  datetime64[ns]
 2   batsman               16515 non-null  object        
 3   batting_team          16515 non-null  object        
 4   bowling_team          16515 non-null  object        
 5   balls_faced           16515 non-null  int64         
 6   fours                 16515 non-null  int64         
 7   sixes                 16515 non-null  int64         
 8   dismissed             16515 non-null  int64         
 9   strike_rate           16515 non-null  float64       
 10  career_matches        16515 non-null  int64         
 11  career_avg_runs       16515 non-null  float64       
 12  form_runs_last_5      16515 non-null  float64       
 13  form_runs_last_1

,matchid,date,batsman,batting_team,bowling_team,balls_faced,fours,sixes,dismissed,strike_rate,career_matches,career_avg_runs,form_runs_last_5,form_runs_last_10,avg_runs_vs_opponent,target_runs
0,548346,2012-04-29,a ashish reddy,deccan chargers,mumbai indians,10,0,1,1,100.0,0,0.00,0.00,0.00,13.500000,10
1,548352,2012-05-04,a ashish reddy,deccan chargers,chennai super kings,3,0,0,1,100.0,1,10.00,10.00,10.00,15.000000,3
2,548359,2012-05-08,a ashish reddy,deccan chargers,kings xi punjab,8,1,0,1,100.0,2,6.50,6.50,6.50,12.333333,8
3,548373,2012-05-18,a ashish reddy,deccan chargers,rajasthan royals,4,2,0,0,250.0,3,7.00,7.00,7.00,12.333333,10
4,548376,2012-05-20,a ashish reddy,deccan chargers,royal challengers bangalore,5,0,0,1,80.0,4,7.75,7.75,7.75,11.000000,4
